# 05 — Model Development (Phase 5)

First modeling pass on the Bank Marketing dataset, using the realistic 14-feature set finalized in `02_feature_leakage_audit.ipynb` / `03_preprocessing.ipynb` — **`duration` is excluded**, since it is only known after a call ends.

Three classifiers are trained and evaluated **separately** on each of the two saved split strategies (stratified random, chronological):

1. **Majority-class baseline** (`DummyClassifier`) — the floor any real model must beat.
2. **Logistic Regression** — with the scaled preprocessing pipeline.
3. **Random Forest** — with the unscaled preprocessing pipeline.

For every (split strategy × model) combination, preprocessing and the model are **fit on training data only** and **evaluated on validation data only**. The test set is never touched in this notebook, and no hyperparameters are tuned — these are first-pass, fixed configurations with fixed random seeds, meant as honest baselines rather than optimized models.

In [1]:
import os
import json

import numpy as np
import pandas as pd
from joblib import dump

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    accuracy_score,
    confusion_matrix,
)

RANDOM_STATE = 42
RESULTS_DIR = "../results"
MODELS_DIR = f"{RESULTS_DIR}/models"
os.makedirs(MODELS_DIR, exist_ok=True)

pd.set_option("display.max_columns", 100)

## Load the dataset and the saved split indices (read-only)

In [2]:
CSV_NAME = "bank-additional-full.csv"

try:
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive")
except ImportError:
    pass  # not running on Colab

_candidates = [
    f"../data/{CSV_NAME}",
    f"data/{CSV_NAME}",
    f"/content/drive/MyDrive/bank-marketing-leakage-audit/data/{CSV_NAME}",
    f"/content/drive/MyDrive/{CSV_NAME}",
]
DATA_PATH = next((p for p in _candidates if os.path.exists(p)), None)

if DATA_PATH is None:
    try:
        from google.colab import files  # type: ignore

        print(f"'{CSV_NAME}' not found in: {_candidates}")
        uploaded = files.upload()
        DATA_PATH = next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError(f"Could not find '{CSV_NAME}' in any of: {_candidates}")

df_raw = pd.read_csv(DATA_PATH, sep=";")

split_paths = {
    "stratified": f"{RESULTS_DIR}/split_stratified.csv",
    "chronological": f"{RESULTS_DIR}/split_chronological.csv",
}
for name, path in split_paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} not found — run 03_preprocessing.ipynb first to generate it.")

splits = {name: pd.read_csv(path) for name, path in split_paths.items()}

print(f"Dataset: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
for name, table in splits.items():
    print(f"{name} split sizes:", table["split"].value_counts().to_dict())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset: 41188 rows, 21 columns
stratified split sizes: {'train': 28831, 'test': 6179, 'val': 6178}
chronological split sizes: {'train': 28831, 'test': 6179, 'val': 6178}


## Rebuild the realistic 14-feature working dataset

This repeats the Phase-3 feature selection and `pdays` recoding exactly (13 approved features + target, `pdays` split into `previously_contacted` / `days_since_previous_contact`), so this notebook is self-contained. `duration` is never selected, so it cannot leak into any model here. `df_raw` itself is not modified.

In [3]:
PRIMARY_FEATURES = [
    "age", "job", "marital", "education", "default", "housing", "loan",
    "pdays", "previous", "poutcome",
    "contact", "month", "day_of_week",
]
TARGET = "y"
assert len(PRIMARY_FEATURES) == 13
assert "duration" not in PRIMARY_FEATURES

data = df_raw[PRIMARY_FEATURES + [TARGET]].copy()
data["previously_contacted"] = (data["pdays"] != 999).astype(int)
data["days_since_previous_contact"] = data["pdays"].replace(999, np.nan)
data = data.drop(columns=["pdays"])

numerical_features = ["age", "previous", "days_since_previous_contact", "previously_contacted"]
categorical_features = ["job", "marital", "education", "default", "housing", "loan",
                         "poutcome", "contact", "month", "day_of_week"]
FEATURE_COLUMNS = numerical_features + categorical_features

assert len(FEATURE_COLUMNS) == 14
assert "duration" not in data.columns

print(f"Working dataset shape: {data.shape} ({len(FEATURE_COLUMNS)} features + target)")
data.head()

Working dataset shape: (41188, 15) (14 features + target)


,age,job,marital,education,default,housing,loan,previous,poutcome,contact,month,day_of_week,y,previously_contacted,days_since_previous_contact
0,56,housemaid,married,basic.4y,no,no,no,0,nonexistent,telephone,may,mon,no,0,NaN
1,57,services,married,high.school,unknown,no,no,0,nonexistent,telephone,may,mon,no,0,NaN
2,37,services,married,high.school,no,yes,no,0,nonexistent,telephone,may,mon,no,0,NaN
3,40,admin.,married,basic.6y,no,no,no,0,nonexistent,telephone,may,mon,no,0,NaN
4,56,services,married,high.school,no,no,yes,0,nonexistent,telephone,may,mon,no,0,NaN


## Preprocessing + model pipelines

Same preprocessing strategy as `03_preprocessing.ipynb` (median imputation for numerics, most-frequent + one-hot for categoricals, `handle_unknown="ignore"`), with `StandardScaler` only for Logistic Regression. Each pipeline bundles preprocessing and the estimator together via `sklearn.pipeline.Pipeline`, so calling `.fit(X_train, y_train)` fits *both* the preprocessing and the model on training data only, and `.predict`/`.predict_proba` on validation data only ever transforms (never re-fits).

Fixed configurations, fixed seeds, no tuning:
- **Majority-class baseline**: `DummyClassifier(strategy="most_frequent")`.
- **Logistic Regression**: default regularization, `max_iter=1000` (for convergence), `random_state=42`.
- **Random Forest**: `n_estimators=100`, `random_state=42`, default depth/split settings.

In [4]:
def make_categorical_pipeline():
    return Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])


def make_preprocessor(scale_numeric):
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))
    return ColumnTransformer(transformers=[
        ("num", Pipeline(steps=numeric_steps), numerical_features),
        ("cat", make_categorical_pipeline(), categorical_features),
    ])


def make_model_pipelines():
    """Fresh, unfit pipelines for one (split) experiment — never reused across splits."""
    return {
        "majority_baseline": Pipeline(steps=[
            ("preprocess", make_preprocessor(scale_numeric=False)),
            ("clf", DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)),
        ]),
        "logistic_regression": Pipeline(steps=[
            ("preprocess", make_preprocessor(scale_numeric=True)),
            ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
        ]),
        "random_forest": Pipeline(steps=[
            ("preprocess", make_preprocessor(scale_numeric=False)),
            ("clf", RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)),
        ]),
    }


make_model_pipelines()

{'majority_baseline': Pipeline(steps=[('preprocess',
                  ColumnTransformer(transformers=[('num',
                                                   Pipeline(steps=[('imputer',
                                                                    SimpleImputer(strategy='median'))]),
                                                   ['age', 'previous',
                                                    'days_since_previous_contact',
                                                    'previously_contacted']),
                                                  ('cat',
                                                   Pipeline(steps=[('imputer',
                                                                    SimpleImputer(strategy='most_frequent')),
                                                                   ('onehot',
                                                                    OneHotEncoder(handle_unknown='ignore'))]),
                                     

## Evaluation helper

A single function runs one (split strategy × model) experiment: it slices train/val rows from the already-saved split table, fits the pipeline on train only, predicts on validation only, and computes every requested metric plus the validation positive-class prevalence. Positive class is `"yes"` throughout.

In [5]:
POS_LABEL = "yes"


def positive_class_proba(fitted_pipeline, X):
    pos_idx = list(fitted_pipeline.classes_).index(POS_LABEL)
    return fitted_pipeline.predict_proba(X)[:, pos_idx]


def run_experiment(split_name, split_table, model_name, pipeline):
    train_idx = split_table.loc[split_table["split"] == "train", "row_index"]
    val_idx = split_table.loc[split_table["split"] == "val", "row_index"]

    X_train = data.loc[train_idx, FEATURE_COLUMNS]
    y_train = data.loc[train_idx, TARGET]
    X_val = data.loc[val_idx, FEATURE_COLUMNS]
    y_val = data.loc[val_idx, TARGET]

    # Fit ONLY on training data.
    pipeline.fit(X_train, y_train)

    # Evaluate ONLY on validation data (test set is never touched here).
    y_pred = pipeline.predict(X_val)
    y_proba = positive_class_proba(pipeline, X_val)

    tn, fp, fn, tp = confusion_matrix(y_val, y_pred, labels=["no", "yes"]).ravel()

    metrics = {
        "split_strategy": split_name,
        "model": model_name,
        "val_prevalence": float((y_val == POS_LABEL).mean()),
        "average_precision": average_precision_score((y_val == POS_LABEL).astype(int), y_proba),
        "precision": precision_score(y_val, y_pred, pos_label=POS_LABEL, zero_division=0),
        "recall": recall_score(y_val, y_pred, pos_label=POS_LABEL, zero_division=0),
        "f1": f1_score(y_val, y_pred, pos_label=POS_LABEL, zero_division=0),
        "roc_auc": roc_auc_score((y_val == POS_LABEL).astype(int), y_proba),
        "accuracy": accuracy_score(y_val, y_pred),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

    predictions = pd.DataFrame({
        "row_index": val_idx.values,
        "split_strategy": split_name,
        "model": model_name,
        "y_true": y_val.values,
        "y_pred": y_pred,
        "y_proba_yes": y_proba,
    })

    return pipeline, metrics, predictions

## Run all experiments: 2 split strategies × 3 models

Each combination gets a freshly constructed, unfit pipeline (via `make_model_pipelines()`), so no fitted state is ever reused across splits or models.

In [6]:
fitted_pipelines = {}
all_metrics = []
all_predictions = []

for split_name, split_table in splits.items():
    pipelines_for_split = make_model_pipelines()
    for model_name, pipeline in pipelines_for_split.items():
        fitted_pipeline, metrics, predictions = run_experiment(split_name, split_table, model_name, pipeline)
        fitted_pipelines[(split_name, model_name)] = fitted_pipeline
        all_metrics.append(metrics)
        all_predictions.append(predictions)
        print(f"[{split_name:13s} | {model_name:20s}] "
              f"AP={metrics['average_precision']:.4f}  ROC-AUC={metrics['roc_auc']:.4f}  "
              f"F1={metrics['f1']:.4f}  prevalence={metrics['val_prevalence']:.4f}")

comparison_df = pd.DataFrame(all_metrics)
predictions_df = pd.concat(all_predictions, ignore_index=True)

[stratified    | majority_baseline   ] AP=0.1127  ROC-AUC=0.5000  F1=0.0000  prevalence=0.1127
[stratified    | logistic_regression ] AP=0.3989  ROC-AUC=0.7540  F1=0.2674  prevalence=0.1127
[stratified    | random_forest       ] AP=0.3334  ROC-AUC=0.7364  F1=0.3258  prevalence=0.1127
[chronological | majority_baseline   ] AP=0.1065  ROC-AUC=0.5000  F1=0.0000  prevalence=0.1065
[chronological | logistic_regression ] AP=0.1978  ROC-AUC=0.6573  F1=0.0000  prevalence=0.1065
[chronological | random_forest       ] AP=0.1864  ROC-AUC=0.6112  F1=0.0783  prevalence=0.1065


## Comparison table

**Average Precision** is the primary metric (most informative under class imbalance); the rest are reported for context. `val_prevalence` is included on every row so metric values can be read relative to the actual positive-class rate in that validation set — this matters a lot here, since the two split strategies have very different validation prevalences (per `04_temporal_split_audit.ipynb`, the chronological split's distribution is not i.i.d. over time).

In [7]:
column_order = [
    "split_strategy", "model", "val_prevalence",
    "average_precision", "roc_auc", "f1", "precision", "recall", "accuracy",
    "tn", "fp", "fn", "tp",
]
comparison_df = comparison_df[column_order].sort_values(["split_strategy", "average_precision"], ascending=[True, False])
comparison_df

,split_strategy,model,val_prevalence,average_precision,roc_auc,f1,precision,recall,accuracy,tn,fp,fn,tp
4,chronological,logistic_regression,0.106507,0.197828,0.657307,0.000000,0.000000,0.000000,0.893493,5520,0,658,0
5,chronological,random_forest,0.106507,0.186398,0.611205,0.078322,0.491228,0.042553,0.893331,5491,29,630,28
3,chronological,majority_baseline,0.106507,0.106507,0.500000,0.000000,0.000000,0.000000,0.893493,5520,0,658,0
1,stratified,logistic_regression,0.112658,0.398944,0.753992,0.267442,0.701220,0.165230,0.898025,5433,49,581,115
2,stratified,random_forest,0.112658,0.333442,0.736427,0.325843,0.467742,0.250000,0.883457,5284,198,522,174
0,stratified,majority_baseline,0.112658,0.112658,0.500000,0.000000,0.000000,0.000000,0.887342,5482,0,696,0


## Save predictions, trained models, and the comparison table

- `results/model_comparison.csv` — one row per (split strategy, model) with every metric above.
- `results/validation_predictions.csv` — one row per validation example per (split strategy, model), with true label, predicted label, and predicted probability of `"yes"`.
- `results/models/<split_strategy>__<model>.joblib` — the fitted pipeline (preprocessing + model together), for each of the 6 combinations.

In [8]:
comparison_path = f"{RESULTS_DIR}/model_comparison.csv"
predictions_path = f"{RESULTS_DIR}/validation_predictions.csv"

comparison_df.to_csv(comparison_path, index=False)
predictions_df.to_csv(predictions_path, index=False)

for (split_name, model_name), fitted_pipeline in fitted_pipelines.items():
    model_path = f"{MODELS_DIR}/{split_name}__{model_name}.joblib"
    dump(fitted_pipeline, model_path)

print("Saved:", comparison_path, "-", comparison_df.shape)
print("Saved:", predictions_path, "-", predictions_df.shape)
print(f"Saved {len(fitted_pipelines)} fitted pipelines to {MODELS_DIR}/")

Saved: ../results/model_comparison.csv - (6, 13)
Saved: ../results/validation_predictions.csv - (37068, 6)
Saved 6 fitted pipelines to ../results/models/


## Summary

- Trained the majority-class baseline, Logistic Regression, and Random Forest on the realistic 14-feature set (`duration` excluded), separately for the stratified and chronological splits — 6 (split × model) combinations in total.
- Every pipeline's preprocessing and model were fit **only** on that split's training rows; all metrics were computed **only** on validation rows. The test set was not touched.
- `val_prevalence` is reported alongside every metric row because the two split strategies have very different validation-set positive rates (expected, per the temporal drift found in `04_temporal_split_audit.ipynb`) — comparing raw metric values across split strategies without this context would be misleading.
- Saved `results/model_comparison.csv`, `results/validation_predictions.csv`, and 6 fitted pipelines under `results/models/`.

No dataset or split file was modified, no hyperparameters were tuned, and the test set was never evaluated.